# KataGo Human SL 分析

重現 `aicup/analyze_humansl.py` 的分析，輸出每局在 10 個 profile 下的落子機率與每手損失。

- runtime：T4 GPU
- 外部資源：KataGo `v1.18.2`、`b18c384nbt-humanv0`、`kata1-b28c512nbt-s13255194368-d5935380940`，授權見報告的外部資源一節
- Drive 的 `MyDrive/aicup-2026/` 要有 `training/` 的 10 個 CSV 與 `katago/` 的 2 個模型
- Colab Secrets 要有唯讀的 `GH_TOKEN`

## 參數

`MODE` 選 `pilot`、`visits`、`main`。中斷後重跑同一個 `MODE` 會接續。

In [ ]:
MODE = 'main'

OUT_DIR = '/content/drive/MyDrive/aicup-2026/humansl-analysis'
RUNS = {
    'pilot': '--players-per-rank 1 --games-per-player 10 --out %s/pilot-100-v1.jsonl' % OUT_DIR,
    'visits': '--players-per-rank 1 --games-per-player 10 --eval-visits 16 --out %s/pilot-100-v16.jsonl' % OUT_DIR,
    'main': '--players-per-rank 30 --games-per-player 20 --out %s/pilot-6000-v1.jsonl' % OUT_DIR,
}
ARGS = RUNS[MODE]

## 取得程式與資料

In [ ]:
import os, subprocess
from google.colab import drive, userdata

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'


def git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr.replace(userdata.get('GH_TOKEN'), '***'))
        raise SystemExit('git %s 失敗' % args[0])


if not os.path.exists(REPO):
    git('clone', '-q', 'https://%s@github.com/Jx-study/aicup-2026-go.git' % userdata.get('GH_TOKEN'), REPO)
os.chdir(REPO)
git('pull', '-q')
!python -m aicup.colab_setup
!git log --oneline -1
!ls training | wc -l
!nvidia-smi -L

## 安裝 KataGo

In [ ]:
%%bash
mkdir -p /content/katago && cd /content/katago
ZIP=katago-v1.18.2-cuda12.8-cudnn9.8.0-linux-x64.zip
wget -q https://github.com/lightvector/KataGo/releases/download/v1.18.2/$ZIP
unzip -qjo $ZIP
chmod +x katago
ls -la katago analysis_example.cfg
ls -la /content/drive/MyDrive/aicup-2026/katago/

## 分析

In [ ]:
!python -m aicup.analyze_humansl {ARGS} 2>&1 | tee -a {OUT_DIR}/{MODE}.log

## 釋放 runtime

跑完才執行。程式結束後 runtime 不會自動中斷，閒置也會扣額度。

In [ ]:
from google.colab import drive, runtime

drive.flush_and_unmount()
runtime.unassign()